# 03_top_artists

DML: gold_top_artists — Ranked artists by plays, including genres.

In [ ]:
%run ../../tools/config/settings
%run ../../tools/delta/upsert

In [ ]:
dbutils.widgets.text("snapshot_date", "")
snapshot_date = dbutils.widgets.get("snapshot_date")

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

fct   = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.fct_plays")
bph   = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_play_history").select("played_at", "track_id", "artist_ids")
dim_a = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_artists").select("artist_id", "artist_name")
today = F.to_date(F.lit(snapshot_date))

plays_with_artists = (
    fct.join(bph, ["played_at", "track_id"], "left")
    .select("play_id", "played_at", F.explode("artist_ids").alias("artist_id"))
)


def _ranked(label, cutoff):
    src = plays_with_artists.filter(cutoff) if cutoff is not None else plays_with_artists
    return (
        src.groupBy("artist_id").agg(F.count("play_id").alias("play_count"))
        .join(dim_a, "artist_id", "left")
        .withColumn("rank", F.row_number().over(Window.orderBy(F.desc("play_count"))))
        .filter(F.col("rank") <= 50)
        .withColumn("period",        F.lit(label))
        .withColumn("snapshot_date", today)
        .select("snapshot_date", "period", "rank", "artist_id", "artist_name", "play_count")
    )


result = (
    _ranked("7d",  F.col("played_at") >= F.date_sub(today, 7))
    .unionByName(_ranked("30d", F.col("played_at") >= F.date_sub(today, 30)))
    .unionByName(_ranked("all", None))
)

upsert_delta(result, f"{CATALOG}.{GOLD_SCHEMA}.gold_top_artists", ["snapshot_date", "period", "rank"])
display(result.orderBy("period", "rank"))